In [1]:
!pip install segmentation-models-pytorch albumentations -q

import os
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    f.write('{"username":"ahnafnafim","key":"YOUR_KEY"}')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

!kaggle datasets download -d araftahsanpavel/tgif-subset -p /kaggle/working/

import zipfile
with zipfile.ZipFile('/kaggle/working/tgif-subset.zip', 'r') as z:
    z.extractall('/kaggle/working/')

# Free space immediately
os.remove('/kaggle/working/tgif-subset.zip')
print("✓ Done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 6.8 MB/s eta 0:00:00
Dataset URL: https://www.kaggle.com/datasets/araftahsanpavel/tgif-subset
License(s): unknown
100%|███████████████████████████████████████| 8.88G/8.88G [01:01<00:00, 155MB/s]

✓ Done


In [2]:
import os, time, random, json
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler, autocast

import albumentations as A
from albumentations.pytorch import ToTensorV2

import segmentation_models_pytorch as smp
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
IMG_SIZE = 384
BATCH_SIZE = 8
NUM_EPOCHS = 20
LR = 1e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 1.0
POS_WEIGHT = 3.0
NUM_WORKERS = 4
PATIENCE = 7
DEVICE = torch.device('cuda')
USE_AMP = True
DATA_ROOT = '/kaggle/working/subset'
SAVE_DIR = '/kaggle/working'

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"Device: {DEVICE} — {torch.cuda.get_device_name(0)}")

Device: cuda — Tesla T4


In [6]:
class TGIFSubsetDataset(Dataset):
    def __init__(self, root, split='training', transform=None, include_authentic=True):
        self.transform = transform
        self.samples = []

        split_dir = os.path.join(root, split)
        fakes_dir = os.path.join(split_dir, 'fakes')
        masks_dir = os.path.join(split_dir, 'masks')
        images_dir = os.path.join(split_dir, 'images')

        categories = []
        if os.path.isdir(fakes_dir):
            categories = [d for d in os.listdir(fakes_dir)
                         if os.path.isdir(os.path.join(fakes_dir, d))]

        for cat in sorted(categories):
            cat_fakes = os.path.join(fakes_dir, cat)
            cat_masks = os.path.join(masks_dir, cat)
            cat_images = os.path.join(images_dir, cat)

            mask_files = {}
            if os.path.isdir(cat_masks):
                for f in os.listdir(cat_masks):
                    if os.path.splitext(f)[1].lower() in IMG_EXTS:
                        mask_files[f] = os.path.join(cat_masks, f)

            if os.path.isdir(cat_fakes):
                for fake_name in sorted(os.listdir(cat_fakes)):
                    if os.path.splitext(fake_name)[1].lower() not in IMG_EXTS:
                        continue
                    for mask_name, mask_path in mask_files.items():
                        if fake_name.startswith(mask_name):
                            self.samples.append((os.path.join(cat_fakes, fake_name), mask_path, 1))
                            break

            if include_authentic and os.path.isdir(cat_images):
                for f in sorted(os.listdir(cat_images)):
                    if os.path.splitext(f)[1].lower() in IMG_EXTS:
                        self.samples.append((os.path.join(cat_images, f), None, 0))

        n_forged = sum(1 for s in self.samples if s[2] == 1)
        n_auth = sum(1 for s in self.samples if s[2] == 0)
        print(f"  {split}: {len(self.samples)} total ({n_forged} forged, {n_auth} authentic)")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path, label = self.samples[idx]
        image = np.array(Image.open(img_path).convert('RGB'))
        if mask_path is not None:
            mask_img = Image.open(mask_path).convert('L')
            if mask_img.size != (image.shape[1], image.shape[0]):
                mask_img = mask_img.resize((image.shape[1], image.shape[0]), resample=Image.NEAREST)
            mask = np.array(mask_img)
            mask = (mask > 127).astype(np.float32)
        else:
            mask = np.zeros((image.shape[0], image.shape[1]), dtype=np.float32)
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask  = augmented['mask']
        if isinstance(mask, np.ndarray):
            mask = torch.from_numpy(mask)
        mask  = mask.unsqueeze(0).float()
        label = torch.tensor(float(label))
        return image, mask, label


# No augmentation — baseline
train_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2()
])

val_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2()
])

print("Building datasets (no augmentation)...")
train_ds = TGIFSubsetDataset(DATA_ROOT, 'training', transform=train_tf)
val_ds   = TGIFSubsetDataset(DATA_ROOT, 'validation', transform=val_tf)
test_ds  = TGIFSubsetDataset(DATA_ROOT, 'testing', transform=val_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)

batch = next(iter(train_loader))
print(f"\nBatch — Image: {batch[0].shape}, Mask: {batch[1].shape}, Label: {batch[2].shape}")

Building datasets (no augmentation)...
  training: 7559 total (5459 forged, 2100 authentic)
  validation: 1023 total (682 forged, 341 authentic)
  testing: 1029 total (686 forged, 343 authentic)

Batch — Image: torch.Size([8, 3, 384, 384]), Mask: torch.Size([8, 1, 384, 384]), Label: torch.Size([8])


In [7]:
class DiceBCELoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.smooth = smooth

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        preds = torch.sigmoid(logits).view(logits.size(0), -1)
        tgt = targets.view(targets.size(0), -1)
        intersection = (preds * tgt).sum(dim=1)
        union = preds.sum(dim=1) + tgt.sum(dim=1)
        dice_loss = 1.0 - ((2.0 * intersection + self.smooth) / (union + self.smooth)).mean()
        return 0.5 * bce_loss + 0.5 * dice_loss


def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    start = time.time()
    for images, masks, labels in loader:
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad(set_to_none=True)
        with autocast('cuda', enabled=USE_AMP):
            logits = model(images)
            loss = criterion(logits, masks)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * images.size(0)
        preds = (torch.sigmoid(logits) > 0.5).float()
        correct += (preds == masks).sum().item()
        total += masks.numel()
    return total_loss / len(loader.dataset), correct / total, time.time() - start


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    for images, masks, labels in loader:
        images, masks = images.to(device), masks.to(device)
        with autocast('cuda', enabled=USE_AMP):
            logits = model(images)
            loss = criterion(logits, masks)
        total_loss += loss.item() * images.size(0)
        preds = (torch.sigmoid(logits) > 0.5).float()
        correct += (preds == masks).sum().item()
        total += masks.numel()
    return total_loss / len(loader.dataset), correct / total


@torch.no_grad()
def full_evaluation(model, loader, device, threshold=0.5):
    model.eval()
    all_pp, all_pt, all_ip, all_it = [], [], [], []
    for images, masks, labels in loader:
        images = images.to(device)
        with autocast('cuda', enabled=USE_AMP):
            logits = model(images)
        probs = torch.sigmoid(logits).cpu().numpy()
        masks_np, labels_np = masks.numpy(), labels.numpy()
        for i in range(probs.shape[0]):
            pred = (probs[i, 0] > threshold).astype(np.int32).flatten()
            gt = masks_np[i, 0].astype(np.int32).flatten()
            all_pp.append(pred); all_pt.append(gt)
            all_ip.append(1 if pred.max() > 0 else 0)
            all_it.append(int(labels_np[i]))
    pp, pt = np.concatenate(all_pp), np.concatenate(all_pt)
    ip, it = np.array(all_ip), np.array(all_it)
    intersection = ((pp == 1) & (pt == 1)).sum()
    union = ((pp == 1) | (pt == 1)).sum()
    return {
        'pixel': {
            'accuracy': accuracy_score(pt, pp),
            'precision': precision_score(pt, pp, zero_division=0),
            'recall': recall_score(pt, pp, zero_division=0),
            'f1': f1_score(pt, pp, zero_division=0),
            'iou': float(intersection / (union + 1e-8)),
            'dice': float(2 * intersection / (pp.sum() + pt.sum() + 1e-8)),
        },
        'image': {
            'accuracy': accuracy_score(it, ip),
            'precision': precision_score(it, ip, zero_division=0),
            'recall': recall_score(it, ip, zero_division=0),
            'f1': f1_score(it, ip, zero_division=0),
        },
        'pixel_cm': confusion_matrix(pt, pp, labels=[0, 1]),
        'image_cm': confusion_matrix(it, ip, labels=[0, 1]) if len(set(it)) > 1 else np.array([[0]]),
    }

print("✓ Functions ready")

✓ Functions ready


In [8]:
model_name = 'EfficientNetV2L_384_baseline'

print(f"{'='*70}")
print(f"TRAINING: {model_name}")
print(f"Config: 384x384, no augmentation, no pos_weight")
print(f"{'='*70}")

model = smp.Unet(
    encoder_name='tu-tf_efficientnetv2_l', encoder_weights='imagenet',
    in_channels=3, classes=1, activation=None
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
model_size = total_params * 4 / 1024**2
print(f"Parameters: {total_params:,}")
print(f"Trainable: {trainable_params:,}")
print(f"Size: {model_size:.1f} MB")

criterion = DiceBCELoss().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)
scaler = GradScaler('cuda', enabled=USE_AMP)

history = {
    'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [],
    'val_f1': [], 'val_iou': [], 'val_precision': [], 'val_recall': [],
    'epoch_time': []
}

best_val_iou = -1.0
best_val_f1 = -1.0
best_epoch = 0
total_time = 0
best_ckpt_path = f'{SAVE_DIR}/best_{model_name}.pth'

for epoch in range(1, NUM_EPOCHS + 1):
    tl, ta, et = train_one_epoch(model, train_loader, criterion, optimizer, scaler, DEVICE)
    vl, va = validate(model, val_loader, criterion, DEVICE)

    val_eval = full_evaluation(model, val_loader, DEVICE, threshold=0.5)
    val_iou = val_eval['pixel']['iou']
    val_f1 = val_eval['pixel']['f1']

    scheduler.step()
    total_time += et

    history['train_loss'].append(tl)
    history['val_loss'].append(vl)
    history['train_acc'].append(ta)
    history['val_acc'].append(va)
    history['val_f1'].append(val_f1)
    history['val_iou'].append(val_iou)
    history['val_precision'].append(val_eval['pixel']['precision'])
    history['val_recall'].append(val_eval['pixel']['recall'])
    history['epoch_time'].append(et)

    if val_iou > best_val_iou:
        best_val_iou = val_iou
        best_val_f1 = val_f1
        best_epoch = epoch
        torch.save({
            'model_state_dict': model.state_dict(),
            'epoch': epoch,
            'best_val_iou': best_val_iou,
            'best_val_f1': best_val_f1,
        }, best_ckpt_path)
        improved = "YES"
    else:
        improved = "NO"

    print(f"Epoch {epoch:>3}/{NUM_EPOCHS} | "
          f"TrLoss: {tl:.4f} TrAcc: {ta:.4f} | "
          f"VlLoss: {vl:.4f} VlAcc: {va:.4f} | "
          f"ValF1: {val_f1:.4f} ValIoU: {val_iou:.4f} | "
          f"{et:.1f}s | Improved: {improved}")

history['total_time'] = total_time
history['best_epoch'] = best_epoch
history['best_val_iou'] = best_val_iou
history['best_val_f1'] = best_val_f1

print(f"\nTraining complete.")
print(f"Best epoch: {best_epoch}")
print(f"Best val IoU: {best_val_iou:.4f}")
print(f"Best val F1: {best_val_f1:.4f}")
print(f"Total time: {total_time:.0f}s")

TRAINING: EfficientNetV2L_384_baseline
Config: 384x384, no augmentation, no pos_weight


Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


Parameters: 119,739,313
Trainable: 119,739,313
Size: 456.8 MB
Epoch   1/20 | TrLoss: 0.6121 TrAcc: 0.8685 | VlLoss: 0.4419 VlAcc: 0.9517 | ValF1: 0.5604 ValIoU: 0.3893 | 480.4s | Improved: YES
Epoch   2/20 | TrLoss: 0.3807 TrAcc: 0.9699 | VlLoss: 0.3750 VlAcc: 0.9646 | ValF1: 0.6448 ValIoU: 0.4757 | 474.8s | Improved: YES
Epoch   3/20 | TrLoss: 0.2974 TrAcc: 0.9814 | VlLoss: 0.3599 VlAcc: 0.9728 | ValF1: 0.6913 ValIoU: 0.5283 | 474.2s | Improved: YES
Epoch   4/20 | TrLoss: 0.2565 TrAcc: 0.9869 | VlLoss: 0.3505 VlAcc: 0.9736 | ValF1: 0.7078 ValIoU: 0.5478 | 473.1s | Improved: YES
Epoch   5/20 | TrLoss: 0.2316 TrAcc: 0.9898 | VlLoss: 0.3559 VlAcc: 0.9750 | ValF1: 0.7258 ValIoU: 0.5696 | 473.3s | Improved: YES
Epoch   6/20 | TrLoss: 0.1624 TrAcc: 0.9919 | VlLoss: 0.2816 VlAcc: 0.9762 | ValF1: 0.7249 ValIoU: 0.5685 | 471.5s | Improved: NO
Epoch   7/20 | TrLoss: 0.1171 TrAcc: 0.9926 | VlLoss: 0.2710 VlAcc: 0.9783 | ValF1: 0.7391 ValIoU: 0.5861 | 468.3s | Improved: YES
Epoch   8/20 | TrLoss:

In [9]:
ckpt = torch.load(best_ckpt_path, map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
print(f"Loaded best checkpoint from epoch {ckpt['epoch']} (val IoU: {ckpt['best_val_iou']:.4f})")

results = full_evaluation(model, test_loader, DEVICE, threshold=0.5)

print(f"\n{'='*60}")
print(f"TEST SET — {model_name}")
print(f"{'='*60}")
print(f"Pixel Accuracy:  {results['pixel']['accuracy']:.4f}")
print(f"Pixel Precision: {results['pixel']['precision']:.4f}")
print(f"Pixel Recall:    {results['pixel']['recall']:.4f}")
print(f"Pixel F1:        {results['pixel']['f1']:.4f}")
print(f"IoU:             {results['pixel']['iou']:.4f}")
print(f"Dice:            {results['pixel']['dice']:.4f}")
print(f"Image F1:        {results['image']['f1']:.4f}")
print(f"{'='*60}")
print(f"\nParameters: {total_params:,}")
print(f"Model size: {model_size:.1f} MB")
print(f"Total training time: {total_time:.0f}s")
print(f"Avg epoch time: {total_time/NUM_EPOCHS:.1f}s")

Loaded best checkpoint from epoch 11 (val IoU: 0.6156)

TEST SET — EfficientNetV2L_384_baseline
Pixel Accuracy:  0.9781
Pixel Precision: 0.7704
Pixel Recall:    0.6870
Pixel F1:        0.7263
IoU:             0.5703
Dice:            0.7263
Image F1:        0.7084

Parameters: 119,739,313
Model size: 456.8 MB
Total training time: 9395s
Avg epoch time: 469.7s


In [10]:
save_data = {
    'model': model_name,
    'config': '384x384, no augmentation, no pos_weight',
    'test_results': {
        'pixel_metrics': results['pixel'],
        'image_metrics': results['image'],
        'pixel_cm': results['pixel_cm'].tolist(),
        'image_cm': results['image_cm'].tolist(),
    },
    'history': history,
    'model_info': {
        'total_params': total_params,
        'trainable_params': trainable_params,
        'model_size_mb': model_size,
    },
}

with open(f'{SAVE_DIR}/{model_name}_results.json', 'w') as f:
    json.dump(save_data, f, indent=2, default=str)
print(f"✓ Saved {model_name}_results.json")

# Print full results for copy-paste backup
print(f"\n{'='*60}")
print("COPY-PASTE BACKUP:")
print(json.dumps({
    'pixel': results['pixel'],
    'image': results['image'],
    'params': total_params,
    'size_mb': model_size,
    'best_epoch': best_epoch,
    'total_time': total_time,
}, indent=2))
print(f"{'='*60}")

# Cleanup checkpoint to save disk
os.remove(best_ckpt_path)
print("✓ Checkpoint removed to save disk")

del model
torch.cuda.empty_cache()

✓ Saved EfficientNetV2L_384_baseline_results.json

COPY-PASTE BACKUP:
{
  "pixel": {
    "accuracy": 0.9781383748781011,
    "precision": 0.7704380555940594,
    "recall": 0.6870139596932916,
    "f1": 0.7263384231112653,
    "iou": 0.5702758380177753,
    "dice": 0.7263384231112647
  },
  "image": {
    "accuracy": 0.6783284742468416,
    "precision": 0.8953229398663697,
    "recall": 0.5860058309037901,
    "f1": 0.7083700440528634
  },
  "params": 119739313,
  "size_mb": 456.769229888916,
  "best_epoch": 11,
  "total_time": 9394.807649612427
}
✓ Checkpoint removed to save disk
